# Aula 06 · Arquitetura Medallion

## O que é
Pense no caminho de uma **joia**:

| Refino | Camada | O que tem no projeto |
|---|---|---|
| ⛏️ Minério bruto: do jeito que saiu da mina | **Bronze** | Os arquivos como chegaram, só com a data de ingestão e o arquivo de origem |
| 🔩 Metal: limpo e padronizado | **Silver** | Datas, valores e categorias corrigidos, sem duplicidade |
| 💍 Joia: pronta para o cliente | **Gold** | Indicadores de negócio prontos para relatórios |

## Por que importa
É o padrão de organização de pipelines **mais adotado no mercado**, e é a estrutura que o projeto do módulo vai seguir. Cada camada tem uma responsabilidade, então fica fácil saber onde corrigir um problema.

## Como aplicar
O projeto tem um notebook para cada camada, na pasta `Projeto/`:

```
landing (arquivos)  →  01_bronze  →  02_silver  →  03_gold
```

Abra cada notebook e leia as explicações. Depois volte aqui e execute as células abaixo, que rodam as três camadas **em sequência**.

> Repare: rodar uma camada depois da outra "na mão" é exatamente o problema que os **jobs** vão resolver a partir do Agrupamento 2.

In [ ]:
CATALOGO = "growdev"

In [ ]:
dbutils.notebook.run("../Projeto/01_bronze", 900, {"catalogo": CATALOGO})
display(spark.sql(f"SELECT customer_id, birth_date, state, declared_monthly_income, _ingested_at FROM {CATALOGO}.bronze.customers LIMIT 5"))

In [ ]:
dbutils.notebook.run("../Projeto/02_silver", 900, {"catalogo": CATALOGO})
display(spark.sql(f"SELECT customer_id, birth_date, state, declared_monthly_income, _processed_at FROM {CATALOGO}.silver.customers LIMIT 5"))

In [ ]:
dbutils.notebook.run("../Projeto/03_gold", 900, {"catalogo": CATALOGO})
display(spark.sql(f"SELECT * FROM {CATALOGO}.gold.resumo_clientes ORDER BY valor_total_emprestado DESC LIMIT 5"))

### O que mudou em cada etapa?
Compare a coluna `birth_date` e `declared_monthly_income` na bronze (texto, vários formatos) e na silver (data e número). Na gold, a pergunta deixa de ser "qual o dado?" e vira "qual o indicador?".

In [ ]:
%sql
SELECT 'bronze' AS camada, COUNT(*) AS linhas FROM growdev.bronze.customers
UNION ALL SELECT 'silver', COUNT(*) FROM growdev.silver.customers
UNION ALL SELECT 'gold', COUNT(*) FROM growdev.gold.resumo_clientes

## Checklist da aula
- [ ] Sei explicar bronze, silver e gold com a analogia do refino
- [ ] Executei as três camadas do projeto
- [ ] Vi as diferenças do dado em cada camada
- [ ] Sei desenhar um pipeline em camadas Medallion